In [0]:
%sql
-- Databricks - Gold KPI builds
-- File: databricks/gold/02_build_kpis.sql

CREATE DATABASE IF NOT EXISTS gold
LOCATION 'abfss://gold@storagedatalake9105.dfs.core.windows.net/adventureworks/_metastore/gold.db';

-- Monthly KPI
CREATE OR REPLACE TABLE gold.kpi_monthly AS
SELECT
  d.year AS order_year,
  d.month AS order_month,
  d.month_name,
  SUM(f.revenue) AS revenue,
  SUM(f.profit) AS profit,
  ROUND(SUM(f.profit) / NULLIF(SUM(f.revenue), 0) * 100, 2) AS profit_margin_pct,
  COUNT(DISTINCT f.OrderNumber) AS orders,
  SUM(f.OrderQuantity) AS total_qty
FROM gold.fact_sales f
JOIN gold.dim_date d
  ON f.date_key = d.date_key
GROUP BY d.year, d.month, d.month_name;

-- Regional KPI
CREATE OR REPLACE TABLE gold.kpi_region AS
SELECT
  d.year AS order_year,
  t.Continent,
  t.Country,
  t.Region,
  SUM(f.revenue) AS revenue,
  SUM(f.profit) AS profit,
  COUNT(DISTINCT f.OrderNumber) AS orders
FROM gold.fact_sales f
JOIN gold.dim_date d
  ON f.date_key = d.date_key
JOIN gold.dim_territory t
  ON f.TerritoryKey = t.TerritoryKey
GROUP BY d.year, t.Continent, t.Country, t.Region;

-- Product KPI
CREATE OR REPLACE TABLE gold.kpi_top_products AS
SELECT
  p.CategoryName,
  p.SubcategoryName,
  p.ProductName,
  SUM(f.revenue) AS revenue,
  SUM(f.profit) AS profit,
  SUM(f.OrderQuantity) AS qty
FROM gold.fact_sales f
JOIN gold.dim_product p
  ON f.ProductKey = p.ProductKey
GROUP BY p.CategoryName, p.SubcategoryName, p.ProductName;

-- Optional quick checks
SELECT COUNT(*) AS kpi_monthly_rows FROM gold.kpi_monthly;
SELECT COUNT(*) AS kpi_region_rows FROM gold.kpi_region;
SELECT COUNT(*) AS kpi_top_products_rows FROM gold.kpi_top_products;
